# Laya — review stages

Real Bin Baz lessons (audio + written text) → speech-to-text → your engine → review cards.

| Stage | What you review | What it tells you |
|---|---|---|
| **1 · GT check** | quotes I extracted from the written text | can the ground truth be trusted? |
| **2 · GT findings** | engine findings that hit a GT quote, pre-labelled automatically | is my automatic labelling right? + engine **recall** |
| **3 · Missed GT** | GT quotes the engine did not find | why the engine missed (engine fault vs garbled speech) |
| **4 · Other findings** | everything else the engine found | engine **precision** + training data with real negatives |
| **5 · Model check** | nothing — models answer, you compare | which model (JEV / Gemini / base Laya) agrees with you |
| 6 · Data generation | *not built yet* | after you pick a model |

**Answers** — every finding gets two answers, exactly Laya's two questions:
- **Verdict**: ✔ *Real quote* · 📖 *Ayah inside a hadith* (verbatim ayah within a hadith; boundary judged against the ayah) · ≈ *Quoted by meaning* (the sheikh tells the hadith / ayah in his own words, mixed with its words — no boundary) · ✘ *Not a quote* · ? *Unsure*
- **Extra toggles** (optional): *Not the engine's fault* (a cut caused by garbled speech) · *Reference wrong / incomplete* (real quote, but another ayah / riwaya is the right match) · *Sheikh's words inside the quote* (he inserted his own explanation in the middle)
- **Boundary** (only for real quotes): *Complete* (the engine caught everything that was said) · *Missing start* / *Missing end* / *Missing both* (the speaker said more of the quote and the engine cut it off). Only words the speaker **actually said** count — never the rest of the verse he didn't say.

**Cards** — yellow = the detected words · green underline = a quote marked in the written text · green fill = the part of the original verse / hadith that was matched.
Answers save on every click to `data/laya_datagen/labels/`. **↶ Undo** reverts your last click, **◀ Back** / **Skip ▶** move without answering. Re-running a cell resumes at the first unanswered card.

## Setup

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import importlib
from backend.laya_datagen import config, asr as asr_mod, pipeline, stages, cards, benchmark, export, sprint, agent_eval
from backend.laya_datagen.teachers import base as teachers
for _m in (teachers, cards, sprint, agent_eval): importlib.reload(_m)   # picks up code edits without restarting the kernel
from backend.laya_datagen.engine_bridge import EngineBridge
from backend.laya_datagen.sources import binbaz

ENGINE  = "v4"                  # the engine you are judging
ENGINES = ["v3", "v4"]          # all engines run (to see which one caught a missed quote); add "v5" once it exists
HADITH_CORPUS = str(ROOT / "data/corpora/hadith_open_hadith_data.json")

SERIES = {   # Bin Baz audio series that have transcripts (many others are audio-only: تفسير ابن كثير, بلوغ المرام, الوابل الصيب …)
    "riyad":        "https://binbaz.org.sa/audios/series/29/%D8%B1%D9%8A%D8%A7%D8%B6-%D8%A7%D9%84%D8%B5%D8%A7%D9%84%D8%AD%D9%8A%D9%86-%D8%A7%D9%84%D8%B4%D8%B1%D8%AD-%D8%A7%D9%84%D8%A7%D9%88%D9%84",  # رياض الصالحين — hadith commentary
    "tawhid_khuz":  "https://binbaz.org.sa/audios/series/79/%D9%83%D8%AA%D8%A7%D8%A8-%D8%A7%D9%84%D8%AA%D9%88%D8%AD%D9%8A%D8%AF-%D9%84%D8%A7%D8%A8%D9%86-%D8%AE%D8%B2%D9%8A%D9%85%D8%A9",  # كتاب التوحيد لابن خزيمة
    "aqeedah":      "https://binbaz.org.sa/audios/series/40/%D8%AF%D8%B1%D9%88%D8%B3-%D9%81%D9%8A-%D8%A7%D9%84%D8%B9%D9%82%D9%8A%D8%AF%D8%A9",  # دروس في العقيدة
    "phone":        "https://binbaz.org.sa/audios/series/56/%D9%85%D8%AD%D8%A7%D8%B6%D8%B1%D8%A7%D8%AA-%D8%B9%D8%A8%D8%B1-%D8%A7%D9%84%D9%87%D8%A7%D8%AA%D9%81",  # محاضرات عبر الهاتف — lectures
    "misc":         "https://binbaz.org.sa/audios/series/43/%D8%AF%D8%B1%D9%88%D8%B3-%D9%85%D9%86%D9%88%D8%B9%D8%A9",  # دروس منوعة
    "radio":        "https://binbaz.org.sa/audios/series/20/%D8%A7%D8%AD%D8%A7%D8%AF%D9%8A%D8%AB-%D9%85%D9%86-%D8%A7%D9%84%D8%A7%D8%B0%D8%A7%D8%B9%D8%A9",  # أحاديث من الإذاعة — short talks
}
LESSONS = {"riyad": 5, "tawhid_khuz": 2, "aqeedah": 2, "phone": 2, "misc": 2, "radio": 3}   # usable lessons per series (with text)
RUN_MODELS = False   # True: call AI models (JEV, Gemini, Laya) for cards they haven't answered yet · False: cached answers only (free, safe for Run all)

## Load lessons
Downloads, transcribes (Groq) and processes each lesson. Everything is cached: re-running only processes new lessons, or all of them after an engine edit (detection only, no new transcription).

In [ ]:
bridge = EngineBridge(hadith_path=HADITH_CORPUS); bridge.start()
bundles, rows = {}, []
for series, n in LESSONS.items():
    for lesson in binbaz.usable_lessons(SERIES[series], n):        # skips audio-only lessons
        try:
            audio = binbaz.download_mp3(lesson)
            asr = asr_mod.transcribe(audio, backend="groq", model="whisper-large-v3-turbo")
            b = pipeline.process_cached(lesson, asr, bridge, ENGINES)
        except Exception as e:                                       # e.g. Groq rate limit: re-run later, the rest is cached
            print(f"✘ {series} {lesson.title[:40]}: {type(e).__name__}: {str(e)[:120]}")
            continue
        rep = b["report"]
        rows.append({"series": series, "lesson": lesson.title[:45], "minutes": round(asr["words"][-1]["end"] / 60),
                     "text↔speech match": f"{min(rep['coverage_asr'], rep['coverage_gt']):.0%}", "usable": rep["passed_gate"],
                     "GT verses": rep["gt_quran_spans"], "GT hadith": rep["gt_hadith_regions"],
                     **{f"{e} findings": sum(r["engine"] == e for r in b["records"]) for e in ENGINES}})
        if rep["passed_gate"]:
            bundles[lesson.id] = b
st = stages.build(bundles, ENGINE, ENGINES)
display(pd.DataFrame(rows))
print(f"{len(bundles)} usable lessons · {len(st['gt'])} GT quotes · {ENGINE}: {len(st['stage2'])} GT findings, "
      f"{len(st['stage3'])} missed GT, {len(st['stage4'])} other findings")

✘ phone أحكام الحج: HTTPStatusError: Client error '404 Not Found' for url 'https://files.zadapps.info/binbaz.org.sa/sawtyaat/dros&amp;mohadrat/mo7adrat_phone


,series,lesson,minutes,text↔speech match,usable,GT verses,GT hadith,v3 findings,v4 findings
0,riyad,01 من: ( باب الإِخلاصِ وإحضار النيَّة في جميع,67,80%,True,53,10,94,89
1,riyad,02 من: ( باب التوبة ),64,73%,True,38,9,74,85
2,riyad,03 من حديث (أن امرأة من جهينة أتت رسول الله ﷺ,78,77%,True,54,18,82,101
3,riyad,04 من حديث (إذا أراد الله بعبده خيرا عجل له ا,72,76%,True,39,18,86,87
4,riyad,05 من حديث (إنكم لتعملون أعمالا هي أدق في أعي,74,77%,True,60,17,85,98
5,tawhid_khuz,01 من بداية مُقدمة المؤلف رحمه الله,23,70%,True,12,0,25,28
6,tawhid_khuz,02 من قوله: (بَاب ذكر إثبات العلم لله جل وعلا,40,61%,True,22,0,56,47
7,aqeedah,التوحيد فضله وأنواعه وحكمه,60,77%,True,75,0,72,68
8,aqeedah,عقيدة أهل السنة والجماعة,91,75%,True,110,0,72,74
9,phone,أحكام النكاح,16,72%,True,3,0,10,8


15 usable lessons · 583 GT quotes · v4: 358 GT findings, 133 missed GT, 374 other findings


---
## Stage 1 · GT check
Is the highlighted part of the **written text** really a quote of the shown verse / hadith?
- **✔ Correct** — right place, right verse, right length
- **✘ Wrong span** — it is a quote, but the highlight starts/ends at the wrong word
- **✘ Wrong verse / hadith** — it is a quote of something else
- **✘ Not a quote** — the highlighted words are ordinary speech
- **? Unsure**

100 random cards. If *Correct* stays above ~97 %, the ground truth is trustworthy and stages 2–3 mean something.

In [ ]:
S1 = st["gt"][:100]
cards.Reviewer(S1, "stage1_gt_check", key=lambda it: it["key"], mode="choice", choices=stages.GT_CHECK,
               render=lambda it: cards.render_gt(it, bundles[it["_bundle"]]), summary=stages.summary_gt)

---
## Stage 2 · GT findings — recall, then check my automatic labels
**Recall** = of all GT quotes that could be matched to the speech, how many the engine found (automatic, no review needed).

In [ ]:
pd.DataFrame(stages.recall_table(bundles, ENGINES))

,engine,type,GT quotes,alignable,caught,recall
0,v3,QURAN,510,428,248,0.579
1,v3,HADITH,73,71,69,0.972
2,v3,ALL,583,499,317,0.635
3,v4,QURAN,510,428,296,0.692
4,v4,HADITH,73,71,70,0.986
5,v4,ALL,583,499,366,0.733


Each card below is an engine finding that lands on a GT quote. The verdict and boundary are **pre-filled with my automatic label** —
if they are right just press **Save & next**; otherwise change them first. The score shows how often my automatic labels agree with you.

In [ ]:
cards.Reviewer(st["stage2"], "findings", key=stages.finding_key, mode="finding", prefill=stages.auto_answer,
               render=lambda r: cards.render_finding(r, bundles[r["_bundle"]], show_auto=True), summary=stages.summary_stage2)

---
## Stage 3 · Missed GT — why didn't the engine find it?
Each card is a GT quote with the speech-to-text at that spot (and audio). The header says whether another engine caught it.
- **Engine should catch it** — the words are recognisable, the engine failed
- **ASR too garbled** — speech-to-text destroyed it; not the engine's fault
- **Bad GT** — the written text was wrong (not really said / not a quote)
- **Too short to matter** — a 2–3 word fragment nobody needs captioned

In [ ]:
cards.Reviewer(st["stage3"], "stage3_missed", key=lambda it: f"{it['key']}|{ENGINE}", mode="choice", choices=stages.MISS_REASON,
               render=lambda it: cards.render_gt(it, bundles[it["_bundle"]], with_asr=True, engine=ENGINE),
               summary=lambda items, ans: stages.summary_missed(items, ans, ENGINE))

### Engine report (updates as you review stages 2–4)

In [ ]:
reps = []
for e in ENGINES:
    st_e = stages.build(bundles, e, ENGINES)
    reps.append(stages.engine_report(st_e, e, cards.load_answers("findings"), cards.load_answers("stage3_missed")))
pd.DataFrame(reps).set_index("engine").T

engine,v3,v4
GT quotes (alignable),499,499
caught,317,366
recall,0.635,0.733
misses reviewed,0/182,133/133
real quotes complete,69/97 (71.1%),151/282 (53.5%)
cut short by the engine,16/97 (16.5%),89/282 (31.6%)
cut short by the ASR (not engine),12/97 (12.4%),42/282 (14.9%)
real quotes with a wrong / incomplete reference,1/97 (1.0%),9/282 (3.2%)
findings,719,732
GT-matched findings real quotes (incl. by meaning),304/308 (98.7%),356/357 (99.7%)


---
## Stage 4 · Other findings — judge the engine, create negatives for Laya
Findings that do **not** land on a marked quote: the written text can't decide them, you do. Nothing is pre-filled (no bias).
The full original verse / hadith is shown with the matched part highlighted, plus the written text at that spot when it could be aligned.

In [ ]:
cards.Reviewer(st["stage4"], "findings", key=stages.finding_key, mode="finding",
               render=lambda r: cards.render_finding(r, bundles[r["_bundle"]]), summary=stages.summary_stage4)

Save everything you labelled in stages 2 and 4 as Laya training rows (split by lesson). Re-run any time.

In [ ]:
human = stages.human_records(bundles, cards.load_answers("findings"), ENGINE)
out = ROOT / "data/laya_datagen/datasets/human"
m = export.export_dataset(human, out, engine=ENGINE, max_per_unit_per_doc=10**6, max_per_unit=10**6)
print(len(human), "labelled findings →", out); pd.DataFrame(m["counts"]).fillna(0).astype(int)

329 labelled findings → /home/zeno/Documents/work/new_projects/Wahy-Transcriber/data/laya_datagen/datasets/human


,train,val,test
FALSE,15,10,1
_rows,240,66,2
CORRECT,225,56,1
boundary:COMPLETE_SPAN,112,38,1
boundary:MISSING_START,36,7,0
boundary:MISSING_END,53,7,0
boundary:MISSING_BOTH,24,4,0


---
## Stage 5 · Model check
Every finding you labelled (stages 2 + 4, *Unsure* excluded) is sent to each model **exactly as the app sends it**, and the model's answer is compared with yours.
- **keeps real quotes** — of your *Real quote* cards, how many the model accepted
- **rejects false findings** — of your *Not a quote* cards, how many it rejected (the hard and important part)
- **calibration error** — 0 = its confidence is honest

Base Laya (no fine-tuning) is the baseline any fine-tuned Laya must beat. Answers are cached. With `RUN_MODELS = False` (Setup) only cached answers are used: nothing is called or paid.

Run the setup cell, then **any model cell on its own**; each one adds its row to the score table. Re-running the setup cell (e.g. after labelling more cards) clears the table: re-run the model cells, cached answers cost nothing.

In [ ]:
from backend.laya_datagen.teachers.jev import JevTeacher
from backend.laya_datagen.teachers.gemini import GeminiTeacher
from backend.laya_datagen.teachers.openrouter_chat import OpenRouterChatTeacher
from backend.laya_datagen.teachers.laya_local import LayaLocal

fa = cards.load_answers("findings")
# models can only answer real / not / unsure: "ayah inside a hadith" counts as real; "quoted by meaning" and "unsure" are left out
gold_recs = [r for r in st["stage2"] + st["stage4"] if stages.finding_key(r) in fa and fa[stages.finding_key(r)]["answer"]["verdict"] in ("CORRECT", "IN_HADITH", "FALSE")]
gold_v = ["FALSE" if fa[stages.finding_key(r)]["answer"]["verdict"] == "FALSE" else "CORRECT" for r in gold_recs]   # ayah inside a hadith = a real quote
gold_b = [fa[stages.finding_key(r)]["answer"].get("boundary") for r in gold_recs]
print(len(gold_recs), "labelled findings (", gold_v.count("CORRECT"), "real /", gold_v.count("FALSE"), "not )")
preds, MODELS = {}, {}

def run_model(name, model):
    """Answer every labelled card with one model (cached), then show the scores of every model run so far."""
    p = teachers.ask(model, [r["envelope"] for r in gold_recs], name, RUN_MODELS)
    if p is not None:
        preds[name], MODELS[name] = p, model
    return scores() if preds else None

def scores():
    table = pd.DataFrame({n: benchmark.score(gold_v, gold_b, p) for n, p in preds.items()}).T
    table["cost $"] = [round(MODELS[n].cost_usd, 4) for n in preds]
    return table

308 labelled findings ( 282 real / 26 not )


### JEV
≈ $0.00005 per call.

In [ ]:
run_model("JEV", JevTeacher())

JEV (308 cached): 0it [00:00, ?it/s]

,n,verdict accuracy,keeps real quotes,rejects false findings,says unsure,boundary accuracy,calibration error,avg latency ms,cost $
JEV,308.0,0.929,1.0,0.154,0.0,0.67,0.05,776.0,0.0


### Gemini
Free tier ≈ 12 requests/min, so a few hundred cards take a while.

In [ ]:
run_model("Gemini 3.5 flash-lite", GeminiTeacher("gemini-3.5-flash-lite"))

Gemini 3.5 flash-lite (308 cached): 0it [00:00, ?it/s]

,n,verdict accuracy,keeps real quotes,rejects false findings,says unsure,boundary accuracy,calibration error,avg latency ms,cost $
JEV,308.0,0.929,1.000,0.154,0.0,0.670,0.050,776.0,0.0
Gemini 3.5 flash-lite,308.0,0.945,0.986,0.500,0.0,0.663,0.013,1039.0,0.0


### Laya (base, no fine-tuning)
Runs locally. This is the baseline a fine-tuned Laya must beat.

In [ ]:
run_model("Laya (base)", LayaLocal(hide_engine_fields=True))

Laya (base) (308 cached): 0it [00:00, ?it/s]

,n,verdict accuracy,keeps real quotes,rejects false findings,says unsure,boundary accuracy,calibration error,avg latency ms,cost $
JEV,308.0,0.929,1.000,0.154,0.000,0.670,0.050,776.0,0.0
Gemini 3.5 flash-lite,308.0,0.945,0.986,0.500,0.000,0.663,0.013,1039.0,0.0
Laya (base),308.0,0.532,0.514,0.731,0.039,0.305,0.193,2760.0,0.0


### Other OpenRouter model (optional)

In [ ]:
# run_model("DeepSeek", OpenRouterChatTeacher("deepseek/deepseek-chat"))

### Compare
Every model run so far, side by side, then where each one went wrong.

In [ ]:
assert preds, "run at least one model cell above"
display(scores())
for n, p in preds.items():
    print(n); display(benchmark.confusion(gold_v, p))

,n,verdict accuracy,keeps real quotes,rejects false findings,says unsure,boundary accuracy,calibration error,avg latency ms,cost $
JEV,308.0,0.929,1.000,0.154,0.000,0.670,0.050,776.0,0.0
Gemini 3.5 flash-lite,308.0,0.945,0.986,0.500,0.000,0.663,0.013,1039.0,0.0
Laya (base),308.0,0.532,0.514,0.731,0.039,0.305,0.193,2760.0,0.0


JEV


,model: CORRECT,model: FALSE,model: NOT_SURE
you: real quote,282,0,0
you: not a quote,22,4,0


Gemini 3.5 flash-lite


,model: CORRECT,model: FALSE,model: NOT_SURE
you: real quote,278,4,0
you: not a quote,13,13,0


Laya (base)


,model: CORRECT,model: FALSE,model: NOT_SURE
you: real quote,145,125,12
you: not a quote,7,19,0


### Where models disagree with you
Read-only cards (Back / Skip to browse): your answer and every model's answer with its reason.

In [ ]:
SHOW = ["JEV", "Gemini 3.5 flash-lite"]   # models to compare here; choose from list(preds)
assert preds, "run at least one model cell first"
show = [n for n in SHOW if n in preds]
dis = [i for i, r in enumerate(gold_recs) if any(preds[n][i]["verdict"] != gold_v[i] for n in show)]
def _extra(i):
    rows = "".join(f"<tr><td>{cards.esc(n)}</td><td><b>{cards.esc(stages.VERDICT[preds[n][i]['verdict']])}</b> "
                   f"{max(preds[n][i]['probs'].values()):.0%}</td><td>{cards.esc(stages.BOUNDARY.get(preds[n][i].get('boundary'), ''))}</td>"
                   f"<td class='ar' style='font-size:15px'>{cards.esc(preds[n][i].get('reason', ''))}</td></tr>" for n in show)
    you = stages.VERDICT[gold_v[i]] + (f" · {stages.BOUNDARY[gold_b[i]]}" if gold_b[i] else "")
    return f"<div class='sec'><div class='lbl'>You: <b>{cards.esc(you)}</b></div><table>{rows}</table></div>"
print(len(dis), "cards with at least one disagreement")
cards.Reviewer(dis, "_browse", key=lambda i: f"b{i}", mode="choice", choices={},
               render=lambda i: cards.render_finding(gold_recs[i], bundles[gold_recs[i]["_bundle"]], extra=_extra(i)),
               summary=lambda items, ans: {})

27 cards with at least one disagreement


---
## Stage 6 · Negatives sprint
Goal: many more ✘ *Not a quote* cards for far fewer clicks. Only stage-4 findings you haven't labelled yet are used.
1. **Prepare** — Gemini and JEV answer every card (cached; Gemini free tier ≈ 4 min per 200 cards). If it stops on an error, re-run: finished cards are kept.
2. **Three queues**, one cell each. No card appears in two queues, and a card never moves once queued:
   - **Gemini-chosen** — Gemini says not a quote / unsure, or Gemini and JEV disagree
   - **Suspicious** — ranked by rules learned from your 26 negatives: narrator chains, CANDIDATE band, no cue, very short, written text has no quote there (on the cards you already labelled, the top 20 held 15 negatives)
   - **Random** — a sample of the rest: shows what the other two miss
3. **One click = answer + next card.** Real quotes are saved too (no boundary needed), to test Gemini and JEV on later.

Answers go to `labels/sprint.json`, kept out of the engine report (these queues are not random samples). Model answers are not shown on the cards, so they don't sway you.

In [ ]:
from backend.laya_datagen.teachers.jev import JevTeacher
from backend.laya_datagen.teachers.gemini import GeminiTeacher

In [ ]:
N_SUSPICIOUS, N_RANDOM = 60, 30             # queue sizes; Gemini-chosen takes every card Gemini picks
sp_pool = sprint.pool(st, cards.load_answers("findings"))
envs = [r["envelope"] for r in sp_pool]
sp_preds = {name: teachers.ask(m, envs, name, RUN_MODELS)
            for name, m in (("Gemini", GeminiTeacher("gemini-3.5-flash-lite")), ("JEV", JevTeacher()))}
Q = None if None in sp_preds.values() else sprint.build_queues(sp_pool, sp_preds["Gemini"], sp_preds["JEV"], N_SUSPICIOUS, N_RANDOM)
print(f"{len(sp_pool)} unlabelled stage-4 findings →", {q: len(v) for q, v in Q.items()} if Q else "no queues")

Gemini (202 cached): 0it [00:00, ?it/s]

JEV (202 cached): 0it [00:00, ?it/s]

202 unlabelled stage-4 findings → {'gemini': 22, 'suspicious': 60, 'random': 30}


### Queue 1 · Gemini-chosen
Most likely to be negatives (in stage 5, about 3 of 4 cards Gemini rejected really were). Strongest first.

In [ ]:
sprint.reviewer(Q, "gemini", bundles)

### Queue 2 · Suspicious
Cards Gemini did *not* pick, ranked by the rules. These are where negatives Gemini misses should turn up.

In [ ]:
sprint.reviewer(Q, "suspicious", bundles)

### Queue 3 · Random
A random sample of what's left. Few negatives expected, but it's the honest check on the two queues above.

In [ ]:
sprint.reviewer(Q, "random", bundles)

### Sprint results
Negatives found per queue (*hit rate* = share of reviewed cards that were not quotes), and how Gemini and JEV did on the cards you answered.
Both models' numbers on the Gemini-chosen queue are flattering by construction; judge them on the suspicious and random queues.

In [ ]:
sprint.results(Q, cards.load_answers(sprint.STORE), sp_pool, sp_preds)

queue,gemini,suspicious,random,all
cards,22,60,30,112
reviewed,21,60,30,111
not a quote,13,16,0,29
real,6,26,25,57
by meaning,2,17,5,24
unsure,0,1,0,1
hit rate,13/21 (61.9%),16/60 (26.7%),0/30 (0.0%),29/111 (26.1%)
Gemini: keeps real quotes,2/6 (33.3%),26/26 (100.0%),25/25 (100.0%),53/57 (93.0%)
Gemini: rejects false findings,12/13 (92.3%),0/16 (0.0%),—,12/29 (41.4%)
JEV: keeps real quotes,4/6 (66.7%),26/26 (100.0%),25/25 (100.0%),55/57 (96.5%)


---
## Stage 7 · AI labeller test (Sonnet)
Can a Claude model label findings the way you do? 30 cards you already labelled (10 not a quote, 4 quoted by meaning, 12 real quotes with mixed boundaries, 4 real quotes from the sprint) are written as plain text to a **blind folder** outside the repo — no answers in it — together with the labelling rules (from your decisions and review notes) and a small tool that records each answer and its time. The card → label link stays in the repo.

In [ ]:
BLIND_DIR = Path.home() / "laya_blind_eval"
agent_eval.prepare(st, bundles, BLIND_DIR)      # once: the same 30 cards afterwards (delete the folder to draw new ones)

round r1 already prepared: /home/zeno/laya_blind_eval


**Run the model in a separate, fresh Claude Code session** (this notebook's session has seen your labels):
1. `cd ~/laya_blind_eval && claude`
2. `/model` → **Sonnet 5.5**, effort **high**
3. paste: `Read CLAUDE.md and label all 30 cards.` — if it asks to read anything outside that folder, say no.

Then run the cell below. To test another model or effort on the same cards, move `answers.json` out of the folder and repeat.

In [ ]:
from IPython.display import Markdown
rep = agent_eval.report(BLIND_DIR)               # also saved to data/laya_datagen/agent_eval/report.md
if rep: display(Markdown(rep["markdown"]))

# AI labeller test — round r1

| labeller | cards | same verdict (5 options) | keeps real quotes | rejects false findings | same boundary (both real) | time | tokens in (fresh / cache write / cache read) | tokens out |
|---|---|---|---|---|---|---|---|---|
| Sonnet | 30 | 29/30 | 16/16 | 10/10 | 12/12 | 1.9 min | 68 / 65,315 / 2,099,705 | 13,215 |
| Gemini + guide | 30 | 22/30 | 16/16 | 5/10 | 6/12 | — |  |  |
| JEV + guide | 30 | 20/30 | 16/16 | 5/10 | 6/12 | — |  |  |
| Gemini (app prompt) | 28 | 18/28 | 15/16 | 3/10 | 7/12 | — |  |  |
| JEV (app prompt) | 28 | 18/28 | 16/16 | 2/10 | 7/12 | — |  |  |

## Disagreements with you

| labeller | card | ref | span | you | model | reason |
|---|---|---|---|---|---|---|
| Sonnet | c27 | صحيح البخاري (2215) | اللهم إن كنت تعلم أني فعلت هذا وتعال وجهك فافرج | ≈ Quoted by meaning | ✔ Real quote · Missing end | the cave-men's dua recited verbatim in the retelling; 'عنا' (ASR 'عندما معه' = 'عنا ما نحن فيه') follows the span and is in the reference |
| Gemini + guide | c03 | صحيح مسلم (3005) | رجع عن دينه، فطرحه بالرأس الجبل، فذهبوا به، فلما صعيدوا الجب | ≈ Quoted by meaning | ✔ Real quote · Complete | The speaker narrates the famous story of the boy and the king from Sahih Muslim, and the detected span accurately captures the hadith text quoted verbatim with minor ASR variations. |
| Gemini + guide | c04 | النساء: 136 | بالله وملائكته وكتبه ورسوله واليوم الآخر، | ✘ Not a quote | ✔ Real quote · Complete | The speaker correctly quotes part of verse 136 from Surah An-Nisa (بالله وملائكته وكتبه ورسله واليوم الآخر) and the span captures the entire quoted segment accurately. |
| Gemini + guide | c08 | صحيح البخاري (2272) | والبقار والغنم والرقي كلها من مالك، قال يا عبد الله لا تستهز | ≈ Quoted by meaning | ✔ Real quote · Complete | The speaker narrates the famous story of the three men in the cave from Sahih al-Bukhari, and the detected span accurately captures the core dialogue and the supplication up to the point where they leave safely. |
| Gemini + guide | c09 | النجم: 31 | بالأرض، ليجزي الذين أساءوا بما عملوا ويجزي الذين أحسنوا | ✔ Real quote · Missing both | ✔ Real quote · Missing end | The speaker recites the verse but cuts off just before the final word بالحسنى, which is explicitly shown in the written text as part of the quote. |
| Gemini + guide | c12 | صحيح البخاري (6309) | أفرح بتوبة عبده من أحدكم، سقط على بعيره وقد أضله في أرضٍ فلا | ✔ Real quote · Missing start | ✔ Real quote · Complete | The speaker clearly quotes the famous hadith verbatim, and the detected span accurately captures the complete matn of the hadith matching the reference. |
| Gemini + guide | c13 | جامع الترمذي (1987) | اتق الله حيث ما كنت، وأتبع السيئة الحسن تمحها، وخلق الناس بخ | ✔ Real quote · Missing end | ✔ Real quote · Complete | The speaker accurately quotes the well-known hadith of Mu'adh, and the detected span covers the text from the start of the hadith up to 'بحسن' matching the reference. |
| Gemini + guide | c15 | النساء: 136 | بالله وملائكته وكتبه برسله وباليوم الآخر | ✘ Not a quote | ✔ Real quote · Complete | The speaker quotes the verse fragment matching the articles of faith correctly, and the detected span captures the exact words said. |
| Gemini + guide | c17 | النساء: 74 | يقاتل سبيل الله، فيقتل | ✘ Not a quote | ✔ Real quote · Complete | The speaker is quoting part of the verse from Surah An-Nisa (وَمَن يُقَاتِلْ فِي سَبِيلِ اللَّهِ فَيُقْتَلْ), and the detected span captures the core of the quote accurately. |
| Gemini + guide | c19 | صحيح البخاري (6001) | أي الذنب أعظم، قال أن تجعل لله ندّاً وهو خلقاً، | ✔ Real quote · Missing start | ✔ Real quote · Complete | The speaker quotes the hadith verbatim, and the detected span covers the full matn correctly. |
| Gemini + guide | c21 | رياض الصالحين (6) | جاءني رسول الله صلى الله عليه وسلم يعودني عام حجة الوداع من  | ✔ Real quote · Missing end | ✔ Real quote · Complete | The speaker recites the hadith text verbatim and completely with no missing words or extra text outside the span. |
| Gemini + guide | c23 | صحيح مسلم (8a) | وتؤمن بالله وملايكته وكتبه ورسوله ولهم الآخر | ✘ Not a quote | ✔ Real quote · Complete | The speaker clearly quotes the famous hadith of Jibril regarding the articles of faith (الإيمان), matching the reference verbatim. |
| Gemini + guide | c26 | الإسراء: 110 | فَلَهُ الْأَسْمَعُ الْحُسْنَى | ✘ Not a quote | ✔ Real quote · Complete | The speaker successfully quotes the phrase from Surah Al-Isra: 110 (فله الأسماء الحسنى), matching the detected span. |
| Gemini + guide | c27 | صحيح البخاري (2215) | اللهم إن كنت تعلم أني فعلت هذا وتعال وجهك فافرج | ≈ Quoted by meaning | ✔ Real quote · Complete | The speaker recites the exact phrasing of the hadith supplication verbatim, and the detected span captures the core of the quote accurately. |
| Gemini + guide | c28 | جامع الترمذي (2616) | الْأَمْرِ الْإِسْلَامِ وَعَمُودُهُ الصَّلَاةِ وَلُوَّةُ سَنَ | ✔ Real quote · Missing both | ✔ Real quote · Complete | The speaker quotes the famous hadith correctly, and the detected span covers the core phrase accurately as said. |
| JEV + guide | c03 | صحيح مسلم (3005) | رجع عن دينه، فطرحه بالرأس الجبل، فذهبوا به، فلما صعيدوا الجب | ≈ Quoted by meaning | ✔ Real quote · Complete |  |
| JEV + guide | c04 | النساء: 136 | بالله وملائكته وكتبه ورسوله واليوم الآخر، | ✘ Not a quote | ✔ Real quote · Complete |  |
| JEV + guide | c08 | صحيح البخاري (2272) | والبقار والغنم والرقي كلها من مالك، قال يا عبد الله لا تستهز | ≈ Quoted by meaning | ✔ Real quote · Complete |  |
| JEV + guide | c09 | النجم: 31 | بالأرض، ليجزي الذين أساءوا بما عملوا ويجزي الذين أحسنوا | ✔ Real quote · Missing both | ✔ Real quote · Missing end |  |
| JEV + guide | c11 | صحيح البخاري (3464) | أبرص فقير كان يعرفك، قال المعنى ما عندك كبر عندما ورثت هذا ا | ≈ Quoted by meaning | ✔ Real quote · Missing both |  |
| JEV + guide | c12 | صحيح البخاري (6309) | أفرح بتوبة عبده من أحدكم، سقط على بعيره وقد أضله في أرضٍ فلا | ✔ Real quote · Missing start | ✔ Real quote · Complete |  |
| JEV + guide | c13 | جامع الترمذي (1987) | اتق الله حيث ما كنت، وأتبع السيئة الحسن تمحها، وخلق الناس بخ | ✔ Real quote · Missing end | ✔ Real quote · Complete |  |
| JEV + guide | c15 | النساء: 136 | بالله وملائكته وكتبه برسله وباليوم الآخر | ✘ Not a quote | ✔ Real quote · Complete |  |
| JEV + guide | c17 | النساء: 74 | يقاتل سبيل الله، فيقتل | ✘ Not a quote | ✔ Real quote · Complete |  |
| JEV + guide | c21 | رياض الصالحين (6) | جاءني رسول الله صلى الله عليه وسلم يعودني عام حجة الوداع من  | ✔ Real quote · Missing end | ✔ Real quote · Complete |  |
| JEV + guide | c22 | صحيح البخاري (6424) | ما لعبد المؤمن جزاء إذا أخذت صفية من هذه الدنيا ثم احتسبه إل | ✔ Real quote · Missing end | 📖 Ayah inside a hadith · Complete |  |
| JEV + guide | c23 | صحيح مسلم (8a) | وتؤمن بالله وملايكته وكتبه ورسوله ولهم الآخر | ✘ Not a quote | 📖 Ayah inside a hadith · Complete |  |
| JEV + guide | c26 | الإسراء: 110 | فَلَهُ الْأَسْمَعُ الْحُسْنَى | ✘ Not a quote | ✔ Real quote · Complete |  |
| JEV + guide | c27 | صحيح البخاري (2215) | اللهم إن كنت تعلم أني فعلت هذا وتعال وجهك فافرج | ≈ Quoted by meaning | 📖 Ayah inside a hadith · Missing end |  |
| JEV + guide | c28 | جامع الترمذي (2616) | الْأَمْرِ الْإِسْلَامِ وَعَمُودُهُ الصَّلَاةِ وَلُوَّةُ سَنَ | ✔ Real quote · Missing both | ✔ Real quote · Complete |  |

---
## Stage 8 · Data generation — *not built yet*
After the tests you pick the labeller(s) to trust; then thousands of findings get labelled automatically and Laya is fine-tuned.

In [ ]:
print('test hi')

test hi
